In [1]:
from numpy import rint
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from logger import configure_logging
import logging
from data_loaders.web_search import manual_web_search
from dotenv import load_dotenv
import os
from langchain.tools import tool
from data_loaders.pdf_ingestion import ingest_from_user_uploaded_pdfs, download_pdf_from_url


load_dotenv(override=True)
tavily_api_key = os.getenv("TAVILY_API_KEY")
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

configure_logging()


### class SourceSchema(BaseModel):
- source_id: "Unique identifier for the source",
- url: Valid URL of the source"
- title: "Title of the source"
- source_type:"Type of the source"
- retrieval_timestamp: "ISO-8601 timestamp of retrieval"
- summary: "Short abstract or snippet"
- confidence_score: "Relevance/confidence score (0-10)"


# Start from pdf ingestion

In [2]:
from logger import get_logger, log_tool_call, log_agent_action, log_warning

logger = get_logger(__name__)

In [3]:
from data_loaders.semantic_scholar_loader import paper_search, snippet_search, get_paper_id, get_paper_citations, get_paper_references, get_paper

In [5]:
query = "Deep Fake Detection"
paper_search_results = paper_search(query)


In [7]:
snippet_search_result = snippet_search(query, limit=5)
print(snippet_search_result)

[{'text': 'A Deep Approach to Deep Fake Detection', 'paperId': None, 'title': 'A Deep Approach to Deep Fake Detection'}, {'text': 'Deep Fake Detecting System', 'paperId': None, 'title': 'Deep Fake Detecting System'}, {'text': 'Decoding deception: state-of-the-art approaches to deep fake detection', 'paperId': None, 'title': 'Decoding deception: state-of-the-art approaches to deep fake detection'}, {'text': 'Deep fake detection using deep learning techniques', 'paperId': None, 'title': 'Deep fake detection using deep learning techniques'}, {'text': 'A Survey: Deep Fake Detection', 'paperId': None, 'title': 'A Survey: Deep Fake Detection'}]


In [8]:
title = snippet_search_result[0]['title']

In [9]:
paper_id = get_paper_id(title)
print(f"Paper ID for '{title}': {paper_id}")

[429] Rate limited. Waiting 1s...
Paper ID for 'A Deep Approach to Deep Fake Detection': 4c1a2669a8486a5f462ec4f068c428ce9a53b5c9


In [ ]:
get_paper(paper_id, fields="title,abstract,year,authors") 

{'paperId': '4c1a2669a8486a5f462ec4f068c428ce9a53b5c9',
 'title': 'A Deep Approach to Deep Fake Detection',
 'year': 2024,
 'openAccessPdf': {'url': 'https://ijsrset.com/index.php/home/article/download/IJSRSET2411274/IJSRSET2411274',
  'status': 'GOLD',
  'license': 'CCBY',
  'disclaimer': 'Notice: Paper or abstract available at https://api.unpaywall.org/v2/10.32628/ijsrset2411274?email=<INSERT_YOUR_EMAIL> or https://doi.org/10.32628/ijsrset2411274, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'},
 'authors': [{'authorId': '2299625404', 'name': 'Prof. Dikshendra Sarpate'},
  {'authorId': '2299648743', 'name': 'Abrar Mungi'},
  {'authorId': '2299630250', 'name': 'Shreyash Borkar'},
  {'authorId': '2299625245', 'name': 'Shravani Mane'},
  {'authorId': '2299625366', 'name': 'Kawnain Shaikh'}],
 'abstract': 'In recent months, the proliferation of free deep le

In [20]:
get_paper_references(paper_id)

{'offset': 0,
 'citingPaperInfo': {'openAccessPdf': {'url': 'https://ijsrset.com/index.php/home/article/download/IJSRSET2411274/IJSRSET2411274',
   'status': 'GOLD',
   'license': 'CCBY',
   'disclaimer': 'Notice: Paper or abstract available at https://api.unpaywall.org/v2/10.32628/ijsrset2411274?email=<INSERT_YOUR_EMAIL> or https://doi.org/10.32628/ijsrset2411274, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use.'},
  'title': 'A Deep Approach to Deep Fake Detection',
  'authors': [{'authorId': '2299625404', 'name': 'Prof. Dikshendra Sarpate'},
   {'authorId': '2299648743', 'name': 'Abrar Mungi'},
   {'authorId': '2299630250', 'name': 'Shreyash Borkar'},
   {'authorId': '2299625245', 'name': 'Shravani Mane'},
   {'authorId': '2299625366', 'name': 'Kawnain Shaikh'}]},
 'data': [{'citedPaper': {'paperId': 'df2df4754aaeefb376fac77a50a6b0e3824a89b4',
    'title'

In [21]:
get_paper_citations(paper_id)

{'offset': 0,
 'data': [{'citingPaper': {'paperId': 'a0b16f8e4b8338d05921ae2cceaa0725119d6ded',
    'title': 'AI Safeguards, Generative AI and the Pandora Box: AI Safety Measures to Protect Businesses and Personal Reputation'}},
  {'citingPaper': {'paperId': '3177cf5388c68cb875f49787e4e5fc3cabccacd4',
    'title': 'Deepfake Detectors on the Edge: CNNs, GAN Fingerprints, and the Race for Real-Time Defense'}},
  {'citingPaper': {'paperId': '33ac9c3e9ff0e2e9a64738b0e5c9f721677f0a90',
    'title': 'A Comprehensive Survey of Deepfake Detection Methods, Datasets, and Future Directions'}},
  {'citingPaper': {'paperId': 'a742f2c793b559136922cf4436cca8bea2a95e95',
    'title': 'AI Powered Solutions for Deepfake Identification'}}]}

In [22]:
len(paper_search_results)


5

In [12]:
from data_loaders.pdf_ingestion import ingest_from_user_uploaded_pdfs

ingest_from_user_uploaded_pdfs("Research Paper")

[]

In [ ]:

def download_pdf_from_url(url: str, save_dir: str = "pdf-from-url") -> str:
    """Download a PDF from a URL and persist it to local storage.

    This helper fetches a remote resource with streaming, validates that the
    response appears to be a PDF, and writes the binary content to disk in the
    target directory. It returns the saved file path for later ingestion or
    returns None when the download fails.

    Attributes:
        url (str): HTTP or HTTPS URL pointing to a PDF resource.
        save_dir (str): Local directory where the downloaded file is stored.

    Example:
        saved_path = download_pdf_from_url(
            "https://example.org/paper.pdf",
            save_dir="pdf-from-url"
        )
    """
    os.makedirs(save_dir, exist_ok=True)

    # Extract filename from URL or fallback
    parsed = urlparse(url)
    filename = os.path.basename(parsed.path) or "downloaded.pdf"

    if not filename.endswith(".pdf"):
        filename += ".pdf"

    file_path = os.path.join(save_dir, filename)

    try:
        response = requests.get(url, stream=True, timeout=15)
        response.raise_for_status()

        # Validate content type (important)
        content_type = response.headers.get("Content-Type", "")
        if "pdf" not in content_type.lower():
            raise ValueError(
                f"URL does not appear to be a PDF (Content-Type: {content_type})"
            )

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        logger.info("Successfully downloaded PDF from %s to %s", url, file_path)
        return file_path

    except Exception as e:
        logger.info("Failed to download PDF from %s: %s", url, e)
        return None
